# V-MD3 Rail-SAR Pipeline | 101-position ISO scan

This notebook processes the single ISO target in horizontal polarization (H/H) and its matched 101-position background scan. Set both input folders in Section 1, confirm the rail aperture, and run the cells in order.

The ISO reference is 70 cm slant range at 53° from horizontal. The corresponding forward ground range is 42.127 cm, and the radar-to-target-plane height difference is 55.90 cm. The image assumes the ISO is centered in cross-range. If it was placed off center, adjust its reference coordinate and peak search bounds in Section 1.

The 10 cm aperture and 0–800,000 Galil encoder span are carried over from the source notebook as defaults. Set them to the values used for this acquisition if they differ.

## Processing flowgraph

```mermaid
flowchart TD
 A["Target and background scans"] --> B["Binary parsing and frame checks"]
 B --> C["Per-RX range FFT and zero Doppler"]
 C --> D["Coherent frame average"]
 D --> E["Matched complex subtraction"]
 E --> F["Ground-plane backprojection"]
 F --> G["Four-RX power fusion"]
```

## Pipeline walkthrough

Each scan position is processed independently. If a file has fewer than 20 frames, or has gaps in its DONE counter, the notebook warns and averages the complete RADC frames that are available at that position. Target and background frames are averaged separately, so their repeat counts can differ.

The complex background average is subtracted from the complex target average at the same rail index and RX channel. If either side has no usable RADC frames at a position, that position receives zero aperture weight. The other positions retain their original physical coordinates.

The backprojection uses `R = sqrt((x - x_aperture)^2 + y^2 + h^2)`, with the same fixed height difference for every pixel. The image combines the power of the four independently focused RX channels.

## V-MD3 acquisition geometry

| Parameter | Setting |
|---|---|
| Rail position indices | 0–100, center index 50 |
| Galil sweep | 0–800,000 by default, with 8,000 counts per step |
| Physical aperture | 10 cm by default, with 1 mm per step |
| Polarization | Horizontal transmit and receive (H/H) |
| Repeated frames | 20 expected per position, averaged over available complete RADC frames |
| Windows | Hamming across range samples and aperture, Hann across chirps |
| Range FFT | 1024 points |
| Phase sign | +1 |
| Slant range and angle | 70 cm at 53° from horizontal |
| Forward ground range | `70*cos(53°)` ≈ 42.127 cm |
| Radar-to-target-plane height | `70*sin(53°)` ≈ 55.904 cm |

## 1. Configuration

Set the paths to the folders containing the target and matched background binaries. Set the physical aperture to the measured end-to-end radar travel for this scan.

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Set these to the folders containing the 101 target and 101 background files.
TARGET_BIN_DIR = Path(r"C:\My Computer\vmd3\notebooks\data\2026_11_06\bins\dual_slant_62cm_angle_48deg_vpol")
BACKGROUND_BIN_DIR = Path(r"C:\My Computer\vmd3\notebooks\data\2026_11_06\bins\iso_slant_62cm_angle_48deg_vpol_bg")
TARGET_RUN = "T"
BACKGROUND_RUN = "B"
RUN_DIRS = {TARGET_RUN: TARGET_BIN_DIR, BACKGROUND_RUN: BACKGROUND_BIN_DIR}

SAR_POSITION_INDICES = np.arange(101)
CENTER_POSITION_INDEX = len(SAR_POSITION_INDICES) // 2
EXPECTED_FRAMES_PER_POSITION = 5
GALIL_START_COUNT = 0
GALIL_END_COUNT = 800_000
GALIL_ENCODER_COUNTS = np.linspace(
    GALIL_START_COUNT, GALIL_END_COUNT, len(SAR_POSITION_INDICES)
).round().astype(int)
SAR_APERTURE_LENGTH_M = 0.10  # Prior notebook's span. Set to the actual rail travel.

CARRIER_FREQUENCY_HZ = 61.0e9
RX_ELEMENT_SPACING_M = 2.464e-3
RX_CHANNELS = np.arange(4)
MEASURED_PHASE_SIGN = +1
NOMINAL_MAXIMUM_RANGE_M = 6.0
RANGE_WINDOW = "hamming"
DOPPLER_WINDOW = "hann"
RANGE_FFT_SIZE = 1024
APERTURE_WINDOW = "hamming"

SCENE_CENTER_SLANT_RANGE_M = 0.62
LOOK_ANGLE_FROM_HORIZONTAL_DEG = 48.0
look_angle_rad = np.deg2rad(LOOK_ANGLE_FROM_HORIZONTAL_DEG)
COMMON_HEIGHT_DIFFERENCE_M = SCENE_CENTER_SLANT_RANGE_M * np.sin(look_angle_rad)
SCENE_CENTER_GROUND_RANGE_M = SCENE_CENTER_SLANT_RANGE_M * np.cos(look_angle_rad)

ISO_REFERENCE_CROSS_RANGE_M = 0.0
ISO_PEAK_SEARCH_HALF_WIDTH_M = 0.15
ISO_PEAK_SEARCH_HALF_DEPTH_M = 0.15

IMAGE_CROSS_RANGE_M = np.linspace(-0.30, 0.30, 301)
# Original variable name retained; now means forward ground range y.
IMAGE_DOWNRANGE_M = np.linspace(
    SCENE_CENTER_GROUND_RANGE_M - 0.20,
    0.80,
    291,
)
IMAGE_DISPLAY_FLOOR_DB = -25.0

for run, folder in RUN_DIRS.items():
    print(f"Run {run}: {folder} | exists = {folder.is_dir()}")
print(f"Radar-to-target-plane height: {COMMON_HEIGHT_DIFFERENCE_M*100:.3f} cm")
print(f"Reference ground range: {SCENE_CENTER_GROUND_RANGE_M*100:.3f} cm")

## 2. Binary parsing and manifests

Each binary record starts with a four-byte stream code and a four-byte little-endian payload length. Missing records, incomplete records at the end of a file, and DONE counter gaps produce warnings. The range processing uses complete RADC records. RFFT and DONE are checked for acquisition diagnostics.

In [ ]:
_POSITION_PATTERN = re.compile(
    r"^scan_(?:pos)?(?P<position>\d+)cm\.bin$",
    flags=re.IGNORECASE,
)

FRAME_HEADER_BYTES = 8
STREAM_CODE_TO_NAME = {b"RADC": "radc", b"RFFT": "rfft", b"DONE": "done"}
EXPECTED_PAYLOAD_BYTES = {"radc": 131_072, "rfft": 131_072, "done": 4}
RADC_FRAME_SHAPE = (128, 64, 4)

def parse_sar_position_index(filename: str) -> int:
    """
    Extract the acquisition position index from a scan filename.

    Examples
    --------
    scan_0cm.bin     -> 0
    scan_pos8cm.bin  -> 8
    scan_pos16cm.bin -> 16

    The extracted number is treated as a position index, not as a
    trustworthy physical distance in centimeters.
    """
    match = _POSITION_PATTERN.fullmatch(filename)

    if match is None:
        raise ValueError(
            f"Could not determine position index from filename: {filename}"
        )

    return int(match.group("position"))


def scan_vmd3_bin(path: Path) -> pd.DataFrame:
    """
    Scan one acquisition .bin file and return one row per stored frame.

    Stored frame format
    -------------------
    bytes 0:4   Stream code: RADC, RFFT, or DONE
    bytes 4:8   Payload length, little-endian unsigned integer
    bytes 8:    Payload
    """
    path = Path(path)
    file_size = path.stat().st_size

    rows = []
    stream_counts = {
        stream_name: 0
        for stream_name in STREAM_CODE_TO_NAME.values()
    }

    offset = 0
    record_index = 0

    with path.open("rb") as file:
        while offset < file_size:
            file.seek(offset)
            header = file.read(FRAME_HEADER_BYTES)

            if len(header) != FRAME_HEADER_BYTES:
                warnings.warn(
                    f"{path.parent.name}/{path.name}: incomplete final frame header "
                    f"at byte offset {offset:,}. Using earlier complete records."
                )
                break

            stream_code = header[:4]

            if stream_code not in STREAM_CODE_TO_NAME:
                raise ValueError(
                    f"{path.name}: unknown stream code "
                    f"{stream_code!r} at byte offset {offset:,}."
                )

            stream = STREAM_CODE_TO_NAME[stream_code]

            payload_length = int.from_bytes(
                header[4:8],
                byteorder="little",
                signed=False,
            )

            payload_offset = offset + FRAME_HEADER_BYTES
            next_offset = payload_offset + payload_length
            if next_offset > file_size:
                warnings.warn(
                    f"{path.parent.name}/{path.name}: incomplete final {stream.upper()} "
                    f"record {record_index}. Using earlier complete records."
                )
                break

            expected_length = EXPECTED_PAYLOAD_BYTES[stream]
            if payload_length != expected_length:
                raise ValueError(
                    f"{path.name}: {stream.upper()} record {record_index} has payload "
                    f"length {payload_length:,}; expected {expected_length:,}."
                )

            stream_frame_index = stream_counts[stream]
            stream_counts[stream] += 1

            rows.append(
                {
                    "record_index": record_index,
                    "stream": stream,
                    "stream_frame_index": stream_frame_index,
                    "header_offset": offset,
                    "payload_offset": payload_offset,
                    "payload_length": payload_length,
                    "frame_length": (
                        FRAME_HEADER_BYTES + payload_length
                    ),
                }
            )

            offset = next_offset
            record_index += 1

    return pd.DataFrame(
        rows,
        columns=[
            "record_index", "stream", "stream_frame_index", "header_offset",
            "payload_offset", "payload_length", "frame_length",
        ],
    )


def read_record_payload(
    path: Path,
    payload_offset: int,
    payload_length: int,
) -> bytes:
    """Read one payload from a parsed V-MD3 record."""
    path = Path(path)

    with path.open("rb") as file:
        file.seek(int(payload_offset))
        payload = file.read(int(payload_length))

    if len(payload) != int(payload_length):
        raise EOFError(
            f"{path.name}: requested {payload_length:,} bytes "
            f"at offset {payload_offset:,}, but read "
            f"{len(payload):,} bytes."
        )

    return payload


def decode_radc_2d(payload: bytes) -> np.ndarray:
    """
    Decode one mode-0 RADC payload.

    Returns
    -------
    cube
        Shape: (sample, chirp, channel) = (128, 64, 4)
    """
    expected_bytes = EXPECTED_PAYLOAD_BYTES["radc"]

    if len(payload) != expected_bytes:
        raise ValueError(
            f"Expected {expected_bytes:,} RADC bytes, "
            f"received {len(payload):,}."
        )

    # Stored as:
    # chirp, channel, interleaved [Q0, I0, Q1, I1, ...]
    raw = np.frombuffer(
        payload,
        dtype="<i2",
    ).reshape(64, 4, 256)

    q = raw[:, :, 0::2].astype(np.float64)
    i = raw[:, :, 1::2].astype(np.float64)

    # Convert from (chirp, channel, sample)
    # to (sample, chirp, channel).
    cube = np.transpose(
        i + 1j * q,
        (2, 0, 1),
    )

    if cube.shape != RADC_FRAME_SHAPE:
        raise ValueError(
            f"Decoded RADC shape is {cube.shape}; "
            f"expected {RADC_FRAME_SHAPE}."
        )

    return cube


def decode_done(payload: bytes) -> int:
    """
    Decode one DONE payload as a little-endian unsigned integer.

    The values appear to behave as radar-side frame counters.
    """
    expected_bytes = EXPECTED_PAYLOAD_BYTES["done"]

    if len(payload) != expected_bytes:
        raise ValueError(
            f"Expected {expected_bytes} DONE bytes, "
            f"received {len(payload)}."
        )

    return int.from_bytes(
        payload,
        byteorder="little",
        signed=False,
    )


def _select_stream_records(
    record_manifest: pd.DataFrame,
    run: str,
    stream: str,
    position_index: int | None = None,
) -> pd.DataFrame:
    """Select and order records for one run, stream, and position."""
    run = run.upper()
    stream = stream.lower()

    if run not in RUN_DIRS:
        raise ValueError(f"Unknown run {run!r}. Expected a configured run label.")
    if stream not in EXPECTED_PAYLOAD_BYTES:
        raise ValueError(f"Unknown stream {stream!r}.")
    if position_index not in SAR_POSITION_INDICES:
        raise ValueError(f"Position index {position_index!r} is outside the configured range.")

    records = record_manifest.loc[
        (record_manifest["run"] == run)
        & (record_manifest["position_index"] == position_index)
        & (record_manifest["stream"] == stream)
    ]
    return records.sort_values("stream_frame_index")


def load_radc_stack(
    record_manifest: pd.DataFrame,
    run: str,
    position_index: int | None = None,
) -> np.ndarray:
    """
    Load RADC data with shape:

        (frame, sample, chirp, channel)
    """
    records = _select_stream_records(
        record_manifest,
        run,
        "radc",
        position_index,
    )

    cubes = []

    for record in records.itertuples(index=False):
        payload = read_record_payload(
            record.path,
            record.payload_offset,
            record.payload_length,
        )

        cubes.append(decode_radc_2d(payload))

    if not cubes:
        return np.empty((0, *RADC_FRAME_SHAPE), dtype=np.complex128)
    return np.stack(cubes, axis=0)


def validate_record_manifest(file_manifest, record_manifest):
    """Report missing records and counter gaps, including files with zero records."""
    records_by_file = record_manifest.groupby(["run", "position_index"])
    expected = EXPECTED_FRAMES_PER_POSITION
    for row in file_manifest.itertuples(index=False):
        group = records_by_file.indices.get((row.run, row.position_index), [])
        records = record_manifest.iloc[group]
        counts = records["stream"].value_counts()
        count_text = ", ".join(
            f"{stream.upper()}={counts.get(stream, 0)}"
            for stream in EXPECTED_PAYLOAD_BYTES
        )
        location = f"Run {row.run}, position {row.position_index} ({row.filename})"
        if any(counts.get(stream, 0) != expected for stream in EXPECTED_PAYLOAD_BYTES):
            warnings.warn(f"{location}: expected {expected} per stream; found {count_text}.")

        done_values = records.loc[records["stream"] == "done", "done_value"].to_numpy(
            dtype=np.int64
        )
        if np.any(np.diff(done_values) != 1):
            warnings.warn(f"{location}: nonconsecutive DONE counter values.")

In [ ]:
# Discover and index the target and background scans independently.
position_filename_pattern = re.compile(
    r"pos_(?P<position>\d{3})\.bin",
    flags=re.IGNORECASE,
)

BIN_FILES = {
    run: sorted(folder.glob("*.bin"))
    for run, folder in RUN_DIRS.items()
}

for run, files in BIN_FILES.items():
    if not files:
        raise ValueError(f"No .bin files found for Run {run}. Check {RUN_DIRS[run]}.")

file_rows = []
for run, files in BIN_FILES.items():
    for path in files:
        match = position_filename_pattern.fullmatch(path.name)
        if match is None:
            raise ValueError(
                f"Run {run}: could not read the position index from {path.name}. "
                "Expected a filename like pos_050.bin."
            )

        position_index = int(match.group("position"))
        if position_index not in SAR_POSITION_INDICES:
            raise ValueError(f"Run {run}: unexpected position index in {path.name}.")

        file_rows.append(
            {
                "run": run,
                "acquisition_type": "sar_scan",
                "position_index": position_index,
                "encoder_count": int(GALIL_ENCODER_COUNTS[position_index]),
                "filename": path.name,
                "file_size_bytes": path.stat().st_size,
                "path": path,
            }
        )

FILE_MANIFEST = (
    pd.DataFrame(file_rows)
    .sort_values(["run", "position_index"])
    .reset_index(drop=True)
)
FILE_MANIFEST["position_index"] = FILE_MANIFEST["position_index"].astype("Int64")
FILE_MANIFEST["encoder_count"] = FILE_MANIFEST["encoder_count"].astype("Int64")

expected_positions = set(SAR_POSITION_INDICES.tolist())
for run in RUN_DIRS:
    run_rows = FILE_MANIFEST.loc[FILE_MANIFEST["run"] == run]
    found_positions = set(run_rows["position_index"].astype(int))
    missing = sorted(expected_positions - found_positions)
    if missing:
        raise ValueError(f"Run {run} is missing position files for indices {missing}.")
    if run_rows["position_index"].duplicated().any():
        raise ValueError(f"Run {run} contains duplicate position indices.")

record_parts = []
for file_row in FILE_MANIFEST.itertuples(index=False):
    records = scan_vmd3_bin(file_row.path)
    records.insert(0, "run", file_row.run)
    records.insert(1, "acquisition_type", file_row.acquisition_type)
    records.insert(2, "position_index", file_row.position_index)
    records.insert(3, "encoder_count", file_row.encoder_count)
    records.insert(4, "filename", file_row.filename)
    records.insert(5, "path", file_row.path)
    records["done_value"] = pd.Series(pd.NA, index=records.index, dtype="Int64")

    for index in records.index[records["stream"] == "done"]:
        payload = read_record_payload(
            records.at[index, "path"],
            records.at[index, "payload_offset"],
            records.at[index, "payload_length"],
        )
        records.at[index, "done_value"] = decode_done(payload)

    record_parts.append(records)

RECORD_MANIFEST = pd.concat(record_parts, ignore_index=True)
for column in ("position_index", "encoder_count", "done_value"):
    RECORD_MANIFEST[column] = RECORD_MANIFEST[column].astype("Int64")

validate_record_manifest(FILE_MANIFEST, RECORD_MANIFEST)
display(FILE_MANIFEST.groupby("run").agg(files=("filename", "size")))
display(RECORD_MANIFEST.groupby(["run", "stream"]).size().unstack(fill_value=0))

## 3. Rail geometry

Both folders must contain the same indices, 0–100. The original `scan_0cm.bin` and `scan_pos0cm.bin` patterns are supported. The number in the filename is the index. The physical position comes from the configured rail aperture and encoder counts.

In [ ]:
def build_sar_position_table(
    file_manifest,
    runs,
    aperture_length_m,
    center_position_index,
):
    """
    Build and validate the physical rail-position table.

    Encoder counts from every requested run must agree at
    every position index.
    """

    run_position_tables = {}

    for run in runs:
        run = run.upper()

        run_rows = (
            file_manifest.loc[
                file_manifest["run"] == run,
                [
                    "position_index",
                    "encoder_count",
                ],
            ]
            .drop_duplicates()
            .sort_values(
                "position_index"
            )
            .reset_index(
                drop=True
            )
        )

        if run_rows.empty:
            raise ValueError(
                f"No position records were found for Run {run}."
            )

        run_rows[
            "position_index"
        ] = run_rows[
            "position_index"
        ].astype(int)

        run_rows[
            "encoder_count"
        ] = run_rows[
            "encoder_count"
        ].astype(int)

        run_position_tables[
            run
        ] = run_rows

    # Use the first requested run as the reference position table.
    reference_run = runs[0].upper()

    reference_table = (
        run_position_tables[
            reference_run
        ].copy()
    )

    expected_position_indices = np.asarray(
        SAR_POSITION_INDICES,
        dtype=int,
    )

    if not np.array_equal(
        reference_table[
            "position_index"
        ].to_numpy(),
        expected_position_indices,
    ):
        raise ValueError(
            f"Run {reference_run} does not contain the "
            "expected ordered SAR position indices."
        )

    # Confirm that all requested runs use identical encoder counts.
    for run in runs[1:]:
        run = run.upper()

        comparison_table = (
            run_position_tables[
                run
            ]
        )

        if not np.array_equal(
            comparison_table[
                "position_index"
            ].to_numpy(),
            reference_table[
                "position_index"
            ].to_numpy(),
        ):
            raise ValueError(
                f"Run {run} position indices do not match "
                f"Run {reference_run}."
            )

        if not np.array_equal(
            comparison_table[
                "encoder_count"
            ].to_numpy(),
            reference_table[
                "encoder_count"
            ].to_numpy(),
        ):
            raise ValueError(
                f"Run {run} encoder counts do not match "
                f"Run {reference_run}."
            )

    center_rows = reference_table.loc[
        reference_table[
            "position_index"
        ]
        == int(
            center_position_index
        )
    ]

    if len(center_rows) != 1:
        raise ValueError(
            "The center position index does not identify "
            "exactly one encoder count."
        )

    center_encoder_count = int(
        center_rows.iloc[0][
            "encoder_count"
        ]
    )

    minimum_encoder_count = int(
        reference_table[
            "encoder_count"
        ].min()
    )

    maximum_encoder_count = int(
        reference_table[
            "encoder_count"
        ].max()
    )

    encoder_span = (
        maximum_encoder_count
        - minimum_encoder_count
    )

    if encoder_span <= 0:
        raise ValueError(
            "The encoder-count span must be positive."
        )

    meters_per_encoder_count = (
        aperture_length_m
        / encoder_span
    )

    reference_table[
        "rail_position_m"
    ] = (
        reference_table[
            "encoder_count"
        ]
        - center_encoder_count
    ) * meters_per_encoder_count

    reference_table[
        "rail_position_cm"
    ] = (
        100
        * reference_table[
            "rail_position_m"
        ]
    )

    rail_position_differences_m = np.diff(
        reference_table[
            "rail_position_m"
        ].to_numpy()
    )

    if not np.allclose(
        rail_position_differences_m,
        rail_position_differences_m[0],
    ):
        raise ValueError(
            "The calculated rail positions are not uniformly spaced."
        )

    reference_table.attrs[
        "aperture_length_m"
    ] = aperture_length_m

    reference_table.attrs[
        "center_position_index"
    ] = center_position_index

    reference_table.attrs[
        "center_encoder_count"
    ] = center_encoder_count

    reference_table.attrs[
        "meters_per_encoder_count"
    ] = meters_per_encoder_count

    return reference_table

In [ ]:
sar_position_table = build_sar_position_table(
    file_manifest=FILE_MANIFEST,
    runs=tuple(RUN_DIRS),
    aperture_length_m=SAR_APERTURE_LENGTH_M,
    center_position_index=CENTER_POSITION_INDEX,
)

rail_positions_m = sar_position_table["rail_position_m"].to_numpy(dtype=float)
rail_step_m = float(np.diff(rail_positions_m)[0])
rx_positions_m = (
    np.arange(4) - (4 - 1) / 2
) * RX_ELEMENT_SPACING_M
rx_phase_center_positions_m = (
    rail_positions_m[:, np.newaxis]
    + rx_positions_m[np.newaxis, :]
)

assert rx_phase_center_positions_m.shape == (len(SAR_POSITION_INDICES), len(RX_CHANNELS))
assert np.isclose(rail_positions_m[CENTER_POSITION_INDEX], 0.0)

print(
    f"Rail geometry: {len(rail_positions_m)} positions, "
    f"{rail_step_m * 1000:.2f} mm step, "
    f"{(rail_positions_m[-1] - rail_positions_m[0]) * 100:.1f} cm aperture"
)
display(sar_position_table)

## 4. Range processing, zero Doppler, and coherent frame averaging

For each RADC frame, every RX channel is independently Hamming-weighted across the 128 measured fast-time samples and evaluated on a 1024-point range-FFT grid. A Hann window is applied across the 64 chirps, and the stationary zero-Doppler component is computed by complex summation. The available zero-Doppler profiles at each rail position are coherently averaged without combining receiver channels. Processing one position at a time bounds memory use and allows frame counts to vary.

The repeated-frame coherence used in the code is

$$
\eta_{\mathrm{frame}}(q)=
\frac{\left|\sum_f X_f(q)\right|}
     {\sum_f\left|X_f(q)\right|},
$$

where a value near one indicates that the repeated-frame phases are stable enough for coherent averaging.

In [ ]:
WINDOW_FUNCTIONS = {
    "rectangular": lambda length: np.ones(length),
    "hann": np.hanning,
    "hamming": np.hamming,
    "blackman": np.blackman,
}

def create_window(window_name, length):
    normalized_name = window_name.lower()
    if normalized_name not in WINDOW_FUNCTIONS:
        raise ValueError(
            f"Unsupported window '{window_name}'. "
            f"Choose from {list(WINDOW_FUNCTIONS)}."
        )
    return WINDOW_FUNCTIONS[normalized_name](length)


def process_all_rx_sar_with_oversampled_range_fft(
    sar_radc_stack,
    range_fft_size,
    range_window="hamming",
    doppler_window="hann",
):
    """Range-process all physical RX channels without combining them."""
    if sar_radc_stack.ndim != 5:
        raise ValueError(
            "sar_radc_stack must have shape "
            "(position, frame, sample, chirp, RX)."
        )

    (
        number_of_positions,
        number_of_frames,
        number_of_fast_time_samples,
        number_of_chirps,
        number_of_rx_channels,
    ) = sar_radc_stack.shape

    if range_fft_size < number_of_fast_time_samples:
        raise ValueError(
            "range_fft_size must be at least the number of measured samples."
        )

    range_coefficients = create_window(
        range_window, number_of_fast_time_samples
    )
    doppler_coefficients = create_window(
        doppler_window, number_of_chirps
    )

    # Output axes: position, repeated frame, range bin, RX channel.
    zero_doppler_stack = np.empty(
        (
            number_of_positions,
            number_of_frames,
            range_fft_size,
            number_of_rx_channels,
        ),
        dtype=np.complex128,
    )

    for position_index in range(number_of_positions):
        for frame_index in range(number_of_frames):
            # Input axes: measured sample, chirp, RX channel.
            radc_frame = np.asarray(
                sar_radc_stack[position_index, frame_index],
                dtype=np.complex128,
            )
            windowed_fast_time = (
                radc_frame
                * range_coefficients[:, np.newaxis, np.newaxis]
            )
            range_fft = np.fft.fft(
                windowed_fast_time,
                n=range_fft_size,
                axis=0,
            )
            windowed_chirps = (
                range_fft
                * doppler_coefficients[np.newaxis, :, np.newaxis]
            )
            # The complex chirp sum is the unshifted zero-Doppler bin.
            zero_doppler_stack[position_index, frame_index] = np.sum(
                windowed_chirps,
                axis=1,
            )

    return zero_doppler_stack, number_of_chirps // 2


def calculate_all_rx_frame_coherence(zero_doppler_stack):
    """Repeated-frame coherent-sum efficiency for every range bin and RX."""
    if zero_doppler_stack.ndim != 4:
        raise ValueError(
            "zero_doppler_stack must have shape "
            "(position, frame, range bin, RX)."
        )
    coherent_magnitude = np.abs(np.sum(zero_doppler_stack, axis=1))
    maximum_magnitude = np.sum(np.abs(zero_doppler_stack), axis=1)
    return coherent_magnitude / np.maximum(
        maximum_magnitude,
        np.finfo(float).tiny,
    )


In [ ]:
def load_validate_and_process_run(run):
    """Process one position at a time and average its available RADC frames."""
    print(f"Loading and processing Run {run}...")
    shape = (len(sar_position_table), RANGE_FFT_SIZE, len(RX_CHANNELS))
    coherent_mean = np.zeros(shape, dtype=np.complex128)
    frame_coherence = np.zeros(shape, dtype=float)
    frame_counts = np.zeros(len(sar_position_table), dtype=int)

    for array_index, position in enumerate(sar_position_table.itertuples(index=False)):
        radc = load_radc_stack(RECORD_MANIFEST, run, position.position_index)
        frame_counts[array_index] = radc.shape[0]
        if not radc.shape[0]:
            continue

        zero_doppler, _ = process_all_rx_sar_with_oversampled_range_fft(
            sar_radc_stack=radc[np.newaxis, ...],
            range_fft_size=RANGE_FFT_SIZE,
            range_window=RANGE_WINDOW,
            doppler_window=DOPPLER_WINDOW,
        )
        coherent_mean[array_index] = np.mean(zero_doppler[0], axis=0)
        frame_coherence[array_index] = calculate_all_rx_frame_coherence(zero_doppler)[0]

    print(
        f"  {np.count_nonzero(frame_counts)}/{len(frame_counts)} positions with RADC; "
        f"frame count per position: {frame_counts.min()}–{frame_counts.max()}"
    )
    return {
        "coherent_mean": coherent_mean,
        "frame_coherence": frame_coherence,
        "frame_counts": frame_counts,
    }


RUN_PRODUCTS = {run: load_validate_and_process_run(run) for run in RUN_DIRS}
frame_count_summary = sar_position_table[["position_index", "rail_position_cm"]].copy()
for run in RUN_DIRS:
    frame_count_summary[f"{run} RADC frames"] = RUN_PRODUCTS[run]["frame_counts"]
frame_count_columns = [f"{run} RADC frames" for run in RUN_DIRS]
unusual_frame_counts = frame_count_summary[frame_count_columns].ne(
    EXPECTED_FRAMES_PER_POSITION
).any(axis=1)
if unusual_frame_counts.any():
    display(frame_count_summary.loc[unusual_frame_counts])

## 5. Matched complex background subtraction

Subtract the position-matched background from the target after each scan's coherent frame average. Subtraction is complex and performed independently for each RX channel.

In [ ]:
target = RUN_PRODUCTS[TARGET_RUN]
background = RUN_PRODUCTS[BACKGROUND_RUN]
matched_positions = (target["frame_counts"] > 0) & (background["frame_counts"] > 0)
if not np.all(matched_positions):
    skipped_positions = SAR_POSITION_INDICES[~matched_positions].tolist()
    warnings.warn(f"No usable RADC pair at positions {skipped_positions}; omitting them from focusing.")

residual = target["coherent_mean"] - background["coherent_mean"]
residual[~matched_positions] = 0.0
TARGET_PRODUCTS = {TARGET_RUN: {"residual": residual}}
print(f"Matched complex subtraction used {np.count_nonzero(matched_positions)} "
      f"of {len(matched_positions)} positions.")

## 6. V-MD3 range mapping

The nominal mapping is enabled by default. Set `USE_PROFESSOR_RANGE_CALIBRATION` to `True` to apply the saved per-RX additive corrections. The 70 cm ISO measurement sets the image geometry and reference marker.

In [ ]:
nominal_oversampled_spacing_m = NOMINAL_MAXIMUM_RANGE_M / RANGE_FFT_SIZE

RANGE_CALIBRATION_BY_RX = {
    int(rx_channel): {
        "slope_m_per_bin": nominal_oversampled_spacing_m,
        "intercept_m": 0.0,
        "method": "nominal range mapping",
    }
    for rx_channel in RX_CHANNELS
}

range_calibration_summary = (
    pd.DataFrame.from_dict(RANGE_CALIBRATION_BY_RX, orient="index")
    .rename_axis("rx_channel")
    .reset_index()
)
display(range_calibration_summary)

## 7. Near-field backprojection

Each RX channel uses the same target-plane height and the original +1 phase convention. Hamming aperture weights are applied to the matched positions, and the four focused powers are averaged.

In [ ]:
def backproject_calibrated_rx_channel(
    aperture_range_profiles,
    aperture_positions_m,
    image_cross_range_m,
    image_downrange_m,
    carrier_frequency_hz,
    measured_phase_sign,
    range_slope_m_per_bin,
    range_intercept_m,
    reference_aperture_index,
    aperture_weights,
    height_difference_m=0.0,
):
    '''Form a calibrated single-RX near-field SAR image by complex backprojection.'''
    profiles = np.asarray(aperture_range_profiles, dtype=np.complex128)
    aperture_positions_m = np.asarray(aperture_positions_m, dtype=float)
    weights = np.asarray(aperture_weights, dtype=float)
    if profiles.ndim != 2:
        raise ValueError("Profiles must have shape (aperture position, range bin).")
    if aperture_positions_m.shape != (profiles.shape[0],):
        raise ValueError("Aperture positions do not match the profile stack.")
    if weights.shape != (profiles.shape[0],) or np.any(weights < 0):
        raise ValueError("Aperture weights are invalid.")
    weight_sum = np.sum(weights)
    if weight_sum > 0:
        weights = weights / weight_sum

    wavelength_m = 299_792_458.0 / carrier_frequency_hz
    cross_grid_m, down_grid_m = np.meshgrid(
        image_cross_range_m, image_downrange_m
    )
    coherent_image = np.zeros(cross_grid_m.shape, dtype=np.complex128)
    noncoherent_sum = np.zeros(cross_grid_m.shape, dtype=float)
    valid_support_count = np.zeros(cross_grid_m.shape, dtype=np.int32)

    center_position_m = aperture_positions_m[reference_aperture_index]
    center_pixel_range_m = np.sqrt(
        (cross_grid_m - center_position_m) ** 2 + down_grid_m ** 2 + height_difference_m ** 2
    )

    for aperture_index in np.flatnonzero(weights):
        aperture_position_m = aperture_positions_m[aperture_index]
        pixel_range_m = np.sqrt(
            (cross_grid_m - aperture_position_m) ** 2 + down_grid_m ** 2 + height_difference_m ** 2
        )
        fractional_bin = (pixel_range_m - range_intercept_m) / range_slope_m_per_bin
        lower_bin = np.floor(fractional_bin).astype(np.int64)
        fraction = fractional_bin - lower_bin
        valid = (lower_bin >= 0) & (lower_bin < profiles.shape[1] - 1)
        safe_lower = np.clip(lower_bin, 0, profiles.shape[1] - 2)
        lower_sample = profiles[aperture_index, safe_lower]
        upper_sample = profiles[aperture_index, safe_lower + 1]
        interpolated = (1.0 - fraction) * lower_sample + fraction * upper_sample
        interpolated = np.where(valid, interpolated, 0.0 + 0.0j)

        relative_range_m = pixel_range_m - center_pixel_range_m
        phase_correction = np.exp(
            -1j
            * measured_phase_sign
            * 4.0
            * np.pi
            / wavelength_m
            * relative_range_m
        )
        coherent_image += weights[aperture_index] * interpolated * phase_correction
        noncoherent_sum += weights[aperture_index] * np.abs(interpolated)
        valid_support_count += valid.astype(np.int32)

    focusing_coherence = np.abs(coherent_image) / np.maximum(
        noncoherent_sum, np.finfo(float).tiny
    )
    return coherent_image, focusing_coherence, valid_support_count

def normalized_power_db(
    complex_or_magnitude_data,
    minimum_db=-80.0,
):
    """
    Convert a complex image or a nonnegative magnitude image
    into normalized power in decibels.

    The result is normalized to its own maximum, so the
    strongest displayed pixel is 0 dB.
    """

    # abs() returns magnitude for complex data and leaves a
    # nonnegative magnitude image unchanged.
    magnitude = np.abs(
        complex_or_magnitude_data
    )

    # Radar images are displayed here as normalized power.
    power = magnitude ** 2

    normalized_power = (
        power
        / np.maximum(
            np.max(
                power
            ),
            np.finfo(float).tiny,
        )
    )

    power_db = (
        10
        * np.log10(
            np.maximum(
                normalized_power,
                np.finfo(float).tiny,
            )
        )
    )

    # Apply a lower display floor so extremely small numerical
    # values do not dominate the plot's color scale.
    return np.maximum(
        power_db,
        minimum_db,
    )


def measure_contiguous_minus_3db_width(
    coordinate,
    power_profile,
):
    """
    Measure the contiguous -3 dB width surrounding the
    strongest peak in a one-dimensional power profile.

    This measures the width of the observed image response.
    For an extended target, it should not automatically be
    interpreted as the point-target resolution.
    """

    coordinate = np.asarray(
        coordinate,
        dtype=float,
    )

    power_profile = np.asarray(
        power_profile,
        dtype=float,
    )

    if coordinate.ndim != 1:
        raise ValueError(
            "coordinate must be one-dimensional."
        )

    if power_profile.shape != coordinate.shape:
        raise ValueError(
            "power_profile must match coordinate."
        )

    # Normalize the profile so that its strongest point is
    # exactly 0 dB.
    normalized_power = (
        power_profile
        / np.maximum(
            np.max(
                power_profile
            ),
            np.finfo(float).tiny,
        )
    )

    power_profile_db = (
        10
        * np.log10(
            np.maximum(
                normalized_power,
                np.finfo(float).tiny,
            )
        )
    )

    # Identify the strongest point in the profile.
    peak_index = int(
        np.argmax(
            power_profile
        )
    )

    threshold_db = -3.0

    # Starting at the peak, walk left until the profile falls
    # below -3 dB. This keeps the measurement attached to the
    # main peak instead of accidentally including a separated
    # sidelobe above -3 dB.
    left_inside_index = peak_index

    while (
        left_inside_index > 0
        and power_profile_db[
            left_inside_index - 1
        ]
        >= threshold_db
    ):
        left_inside_index -= 1

    # Repeat the same search toward increasing coordinate.
    right_inside_index = peak_index

    while (
        right_inside_index
        < coordinate.size - 1
        and power_profile_db[
            right_inside_index + 1
        ]
        >= threshold_db
    ):
        right_inside_index += 1

    def interpolate_threshold_crossing(
        coordinate_1,
        level_1,
        coordinate_2,
        level_2,
        threshold,
    ):
        """
        Estimate the coordinate where two adjacent samples
        cross the selected dB threshold.
        """

        if np.isclose(
            level_1,
            level_2,
        ):
            return (
                coordinate_1
                + coordinate_2
            ) / 2

        return (
            coordinate_1
            + (
                threshold
                - level_1
            )
            * (
                coordinate_2
                - coordinate_1
            )
            / (
                level_2
                - level_1
            )
        )

    # Interpolate the left -3 dB crossing. If the profile
    # remains above -3 dB at the grid boundary, the width
    # cannot be measured completely.
    if left_inside_index == 0:
        left_crossing = np.nan
    else:
        left_crossing = (
            interpolate_threshold_crossing(
                coordinate[
                    left_inside_index - 1
                ],
                power_profile_db[
                    left_inside_index - 1
                ],
                coordinate[
                    left_inside_index
                ],
                power_profile_db[
                    left_inside_index
                ],
                threshold_db,
            )
        )

    # Interpolate the right -3 dB crossing.
    if (
        right_inside_index
        == coordinate.size - 1
    ):
        right_crossing = np.nan
    else:
        right_crossing = (
            interpolate_threshold_crossing(
                coordinate[
                    right_inside_index
                ],
                power_profile_db[
                    right_inside_index
                ],
                coordinate[
                    right_inside_index + 1
                ],
                power_profile_db[
                    right_inside_index + 1
                ],
                threshold_db,
            )
        )

    # A valid width requires both threshold crossings.
    if (
        np.isfinite(
            left_crossing
        )
        and np.isfinite(
            right_crossing
        )
    ):
        width = (
            right_crossing
            - left_crossing
        )
    else:
        width = np.nan

    return {
        "peak_index": peak_index,
        "peak_coordinate": coordinate[
            peak_index
        ],
        "left_crossing": left_crossing,
        "right_crossing": right_crossing,
        "width": width,
        "power_profile_db": power_profile_db,
    }


def create_synthetic_aperture_weights(
    window_name,
    number_of_positions,
):
    """
    Create normalized nonnegative amplitude weights for the
    synthetic-aperture positions.

    The returned weights sum to one so that changing the window
    does not introduce an arbitrary coherent-image scale.
    """

    normalized_name = window_name.lower()

    if normalized_name == "uniform":
        raw_weights = np.ones(
            number_of_positions,
            dtype=float,
        )

    elif normalized_name == "hamming":
        raw_weights = np.hamming(
            number_of_positions
        )

    elif normalized_name == "hann":
        raw_weights = np.hanning(
            number_of_positions
        )

    else:
        raise ValueError(
            "Unsupported aperture window. Choose from "
            "'uniform', 'hamming', or 'hann'."
        )

    if not np.any(
        raw_weights > 0
    ):
        raise ValueError(
            "The aperture window contains no positive weights."
        )

    # Normalize the coefficients to unit sum.
    #
    # This preserves image shape and relative coherent gain
    # comparisons without allowing a window with a larger
    # coefficient sum to appear artificially brighter.
    normalized_weights = (
        raw_weights
        / np.sum(
            raw_weights
        )
    )

    return normalized_weights


def characterize_mainlobe_and_outside_response(
    coordinate,
    power_profile,
):
    """
    Characterize one normalized power profile.

    The nearest local minima around the strongest peak define
    the central main-lobe boundaries. The largest value outside
    those boundaries is reported as an out-of-main-lobe level.

    For ideal point-target simulations, this is the ideal peak
    sidelobe level.

    For measured data, it may also contain multipath, clutter,
    or additional scattering centers.
    """

    coordinate = np.asarray(
        coordinate,
        dtype=float,
    )

    power_profile = np.asarray(
        power_profile,
        dtype=float,
    )

    if coordinate.shape != power_profile.shape:
        raise ValueError(
            "coordinate and power_profile must match."
        )

    normalized_power = (
        power_profile
        / np.maximum(
            np.max(
                power_profile
            ),
            np.finfo(float).tiny,
        )
    )

    normalized_power_db = (
        10
        * np.log10(
            np.maximum(
                normalized_power,
                np.finfo(float).tiny,
            )
        )
    )

    peak_index = int(
        np.argmax(
            normalized_power
        )
    )

    # Measure the contiguous -3 dB width surrounding the
    # strongest peak.
    minus3db_result = (
        measure_contiguous_minus_3db_width(
            coordinate=coordinate,
            power_profile=normalized_power,
        )
    )

    # Find local minima that can serve as the boundaries of
    # the central main lobe.
    local_minimum_indices = (
        np.flatnonzero(
            (
                normalized_power[1:-1]
                <= normalized_power[:-2]
            )
            & (
                normalized_power[1:-1]
                < normalized_power[2:]
            )
        )
        + 1
    )

    left_minimum_candidates = (
        local_minimum_indices[
            local_minimum_indices
            < peak_index
        ]
    )

    right_minimum_candidates = (
        local_minimum_indices[
            local_minimum_indices
            > peak_index
        ]
    )

    if (
        left_minimum_candidates.size == 0
        or right_minimum_candidates.size == 0
    ):
        left_mainlobe_minimum_index = None
        right_mainlobe_minimum_index = None
        null_to_null_width = np.nan
        maximum_outside_mainlobe_db = np.nan

    else:
        left_mainlobe_minimum_index = (
            left_minimum_candidates[-1]
        )

        right_mainlobe_minimum_index = (
            right_minimum_candidates[0]
        )

        null_to_null_width = (
            coordinate[
                right_mainlobe_minimum_index
            ]
            - coordinate[
                left_mainlobe_minimum_index
            ]
        )

        # Exclude the region between the first minima and find
        # the largest remaining response.
        outside_mainlobe_mask = np.ones(
            normalized_power.shape,
            dtype=bool,
        )

        outside_mainlobe_mask[
            left_mainlobe_minimum_index:
            right_mainlobe_minimum_index + 1
        ] = False

        maximum_outside_mainlobe_db = np.max(
            normalized_power_db[
                outside_mainlobe_mask
            ]
        )

    return {
        "peak_index": peak_index,
        "peak_coordinate": coordinate[
            peak_index
        ],
        "minus3db_width": (
            minus3db_result[
                "width"
            ]
        ),
        "null_to_null_width": (
            null_to_null_width
        ),
        "maximum_outside_mainlobe_db": (
            maximum_outside_mainlobe_db
        ),
        "power_profile_db": (
            normalized_power_db
        ),
        "left_mainlobe_minimum_index": (
            left_mainlobe_minimum_index
        ),
        "right_mainlobe_minimum_index": (
            right_mainlobe_minimum_index
        ),
    }

In [ ]:
aperture_weights = create_synthetic_aperture_weights(
    window_name=APERTURE_WINDOW,
    number_of_positions=rx_phase_center_positions_m.shape[0],
)
aperture_weights[~matched_positions] = 0.0


def normalized_nonnegative_power_db(power, minimum_db=-60.0):
    """Normalize a power image to a peak of 0 dB."""
    power = np.asarray(power, dtype=float)
    normalized = power / np.maximum(np.max(power), np.finfo(float).tiny)
    power_db = 10.0 * np.log10(
        np.maximum(normalized, np.finfo(float).tiny)
    )
    return np.maximum(power_db, minimum_db)


def form_per_rx_images(run, aperture_phase_corrections=None):
    """Backproject each RX channel using nominal range mapping."""
    channel_images = []
    channel_coherence_maps = []
    range_spacing_m = NOMINAL_MAXIMUM_RANGE_M / RANGE_FFT_SIZE
    active_position_count = np.count_nonzero(aperture_weights)

    for rx_channel in RX_CHANNELS:
        profiles = TARGET_PRODUCTS[run]["residual"][:, :, rx_channel]
        if aperture_phase_corrections is not None:
            profiles = (
                profiles
                * aperture_phase_corrections[:, rx_channel, np.newaxis]
            )

        image, focusing_coherence, support_count = backproject_calibrated_rx_channel(
            aperture_range_profiles=profiles,
            aperture_positions_m=rx_phase_center_positions_m[:, rx_channel],
            image_cross_range_m=IMAGE_CROSS_RANGE_M,
            image_downrange_m=IMAGE_DOWNRANGE_M,
            carrier_frequency_hz=CARRIER_FREQUENCY_HZ,
            measured_phase_sign=MEASURED_PHASE_SIGN,
            range_slope_m_per_bin=range_spacing_m,
            range_intercept_m=0.0,
            reference_aperture_index=CENTER_POSITION_INDEX,
            aperture_weights=aperture_weights,
            height_difference_m=COMMON_HEIGHT_DIFFERENCE_M,
        )

        if not np.all(support_count == active_position_count):
            raise ValueError(
                f"Run {run}, RX {rx_channel} has incomplete aperture support."
            )

        channel_images.append(image)
        channel_coherence_maps.append(focusing_coherence)

    return np.stack(channel_images), np.stack(channel_coherence_maps)


PER_RX_IMAGES, PER_RX_COHERENCE = form_per_rx_images(TARGET_RUN)
FUSED_POWER = np.mean(np.abs(PER_RX_IMAGES) ** 2, axis=0)
FUSED_POWER_DB = normalized_nonnegative_power_db(FUSED_POWER)

print(
    "Equal-power fusion of four independently focused, "
    "background-subtracted RX images."
)

## 8. Final focused image

The dotted line marks the radar's cross-range center. The white circle marks the expected ISO location, and the cyan cross marks the strongest return within the configured search area. The peak and reference can differ because the cylinder can scatter from several locations.

In [ ]:
x_cm = IMAGE_CROSS_RANGE_M * 100
y_cm = IMAGE_DOWNRANGE_M * 100
dx_cm = x_cm[1] - x_cm[0]
dy_cm = y_cm[1] - y_cm[0]
image_extent_cm = [
    x_cm[0] - dx_cm / 2, x_cm[-1] + dx_cm / 2,
    y_cm[0] - dy_cm / 2, y_cm[-1] + dy_cm / 2,
]

# Use unfloored dB values for the numerical readings.
power_db = 10.0 * np.log10(
    np.maximum(FUSED_POWER, np.finfo(float).tiny)
    / np.maximum(np.max(FUSED_POWER), np.finfo(float).tiny)
)

fig, axis = plt.subplots(figsize=(11, 7), constrained_layout=True)

handle = axis.imshow(
    power_db,
    origin="lower",
    extent=image_extent_cm,
    aspect="equal",
    cmap="viridis",
    vmin=IMAGE_DISPLAY_FLOOR_DB,
    vmax=0.0,
)

# Radar's center aperture position.
axis.axvline(
    0.0,
    color="white",
    linestyle=":",
    linewidth=1.8,
    label="Radar center, x = 0 cm",
)

reference_x_cm = ISO_REFERENCE_CROSS_RANGE_M * 100
reference_y_cm = SCENE_CENTER_GROUND_RANGE_M * 100
axis.plot(
    reference_x_cm, reference_y_cm, marker="o", linestyle="none",
    markerfacecolor="none", markeredgecolor="white", markersize=10,
    label=f"ISO reference ({reference_x_cm:.1f}, {reference_y_cm:.3f} cm)",
)

search_region = (
    (np.abs(IMAGE_CROSS_RANGE_M[np.newaxis, :] - ISO_REFERENCE_CROSS_RANGE_M)
     <= ISO_PEAK_SEARCH_HALF_WIDTH_M)
    & (np.abs(IMAGE_DOWNRANGE_M[:, np.newaxis] - SCENE_CENTER_GROUND_RANGE_M)
       <= ISO_PEAK_SEARCH_HALF_DEPTH_M)
)
peak_rows = []
if np.any(search_region & (FUSED_POWER > 0)):
    peak_y_index, peak_x_index = np.unravel_index(
        np.argmax(np.where(search_region, FUSED_POWER, -np.inf)), FUSED_POWER.shape
    )
    peak_x_cm = x_cm[peak_x_index]
    peak_y_cm = y_cm[peak_y_index]
    peak_db = power_db[peak_y_index, peak_x_index]
    axis.plot(
        peak_x_cm, peak_y_cm, marker="x", linestyle="none", color="cyan",
        markersize=10, markeredgewidth=2, label="Peak in ISO search area",
    )
    axis.annotate(
        f"Peak: {peak_db:.2f} dB\nx = {peak_x_cm:.1f}, y = {peak_y_cm:.1f} cm",
        xy=(peak_x_cm, peak_y_cm), xytext=(12, 18), textcoords="offset points",
        color="cyan", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="black", alpha=0.65),
        arrowprops=dict(arrowstyle="-", color="cyan"),
    )
    peak_rows.append({
        "Peak cross-range (cm)": peak_x_cm,
        "Peak ground range (cm)": peak_y_cm,
        "Peak relative power (dB)": peak_db,
    })
else:
    warnings.warn("No nonzero image power in the ISO peak search area.")

axis.set(
    xlabel="Cross-range x (cm)",
    ylabel="Forward ground range y (cm)",
    title="SPHERE | V/V | 101 positions | Matched background subtraction",
)

axis.set_ylim(top=80.0)
axis.legend(loc="upper right", fontsize=8)

fig.colorbar(
    handle,
    ax=axis,
    label="Power relative to whole-image maximum (dB)",
)

plt.show()
if peak_rows:
    display(pd.DataFrame(peak_rows).round(2))

## 9. Reading the image

The plotted forward ground range is measured along the target plane. The 42.127 cm reference comes from the setup geometry. The plotted peak reports the strongest return in the ISO search area. Its dB value is relative to the maximum over the entire fused image.

Frame-count warnings identify the run and position. The table in Section 4 lists positions whose RADC counts differ from the expected 20. Section 5 reports how many positions had usable target and background data.